<img src="https://images.squarespace-cdn.com/content/v1/67351b6c456f5a62d1c1bca2/17a93d03-0cae-49bf-b152-8dfd0cdb9d7d/Quantum+Rings+Logo+Main+White.png?format=300w" alt="Quantum Rings" width="150" align="right"/>

# Grover's Search Algorithm

In 1996 Lov Grover introduced a novel search algorithm which exhibits quadratic speedup and uses a technique called amplitude amplification. Imagine that we are given a printed phone book and asked to find to whom a particular phone number is associated. Phone directories are usually arranged alphabetically. We have to go through each entry in the phone directory to determine whom the phone number belongs.

This sample code implements Grover's algorithm, following the explanation in the Quantum Rings documentation on Grover's Algorithm here:
https://portal.quantumrings.com/doc/Grovers.html

## Required One-time Setup

If you haven't already done so, obtain a free license key from the [Quantum Rings portal](https://portal.quantumrings.com/), then replace the `YOUR_TOKEN_HERE` and `YOUR_EMAIL_ADDRESS` placeholders below with your token and account email.

In [ ]:
%%capture
%pip install QuantumRingsLib matplotlib numpy

In [ ]:
import QuantumRingsLib
from QuantumRingsLib import QuantumRegister, AncillaRegister, ClassicalRegister, QuantumCircuit
from QuantumRingsLib import QuantumRingsProvider
from QuantumRingsLib import job_monitor
from QuantumRingsLib import JobStatus
from matplotlib import pyplot as plt
import numpy as np

provider = QuantumRingsProvider(token="YOUR_TOKEN_HERE", name="YOUR_EMAIL_ADDRESS")
backend = provider.get_backend("scarlet_quantum_rings")
shots = 100

provider.active_account()

In [ ]:
def ccz(qc, control1, control2, target):
    """
    The CCZ gate or the doubly controlled Z-gate.
    This gate flips the phase of the target qubit if the control qubits are in |11> state.

    Args:
         qc (QuantumCircuit):
            The quantum Circuit to use.

        control1 (int):
            The index number of the first control qubit.

        control2 (int):
            The index number of the second control qubit.

        target (int):
            The index number of the target qubit, where the transform is applied

    Returns:
        None


    """
    qc.h(target)
    qc.ccx(control1, control2, target)
    qc.h(target)
    return


def cccx(qc,control1, control2, control3, anc, target):
    """
    The three control Toffoli gate.
    An X gate is applied to the target qubit, if the control qubits are in state |111>.

    Args:
        qc (QuantumCircuit):
            The quantum Circuit to use.

        control1 (int):
            The index number of the first control qubit.

        control2 (int):
            The index number of the second control qubit.

        control3 (int):
            The index number of the third control qubit

        anc (int):
            The index number of a temporary worker qubit

        target (int):
            The index number of the target qubit, where the transform is applied

    Returns:
        None
    """
    qc.ccx(control1, control2, anc)
    qc.ccx(control3, anc, target)
    qc.ccx(control1, control2, anc)
    qc.ccx(control3, anc, target)
    return



def grover_oracle( qc, x0, x1, x2, anc, y):
    """
    The Grover's Oracle

    Args:
        qc (QuantumCircuit):
            The quantum Circuit to use.

        x0 (int):
            The index number of the first qubit of the x register.

        x1 (int):
            The index number of the second qubit of the x register.

        x2 (int):
            The index number of the third qubit of the x register.

        anc (int):
            The index number of a temporary work qubit

        y (int):
            The index number of the target qubit, where the transform is applied

    Returns:
        None.

    """
    qc.x(x2)
    qc.x(x0)
    cccx(qc, x0, x1, x2, anc, y)
    qc.x(x0)
    qc.x(x2)



def grover_diffusion_operator(qc, x0, x1, x2, y):
    """
    The 3-qubit diffusion operator for Grover's algorith.

    Args:
        x0 (int):
            The index number of the first qubit of the x register.

        x1 (int):
            The index number of the second qubit of the x register.

        x2 (int):
            The index number of the third qubit of the x register.

        y (int):
            The index number of the target qubit, where the transform is applied

    Returns:
        None.


    """

    qc.h(x0)
    qc.h(x1)
    qc.h(x2)
    qc.h(y) # Bring this back to state 1 for next stages
    qc.x(x0)
    qc.x(x1)
    qc.x(x2)
    ccz(qc, x0, x1, x2 )
    qc.x(x0)
    qc.x(x1)
    qc.x(x2)
    qc.h(x0)
    qc.h(x1)
    qc.h(x2)



def plot_histogram (counts, title=""):
    """
    Plots the histogram of the counts

    Args:

        counts (dict):
            The dictionary containing the counts of states

        titles (str):
            A title for the graph.

    Returns:
        None

    """
    fig, ax = plt.subplots(figsize =(10, 7))
    plt.xlabel("States")
    plt.ylabel("Counts")
    mylist = [key for key, val in counts.items() for _ in range(val)]

    unique, inverse = np.unique(mylist, return_inverse=True)
    bin_counts = np.bincount(inverse)

    plt.bar(unique, bin_counts)

    maxFreq = max(counts.values())
    plt.ylim(ymax=np.ceil(maxFreq / 10) * 10 if maxFreq % 10 else maxFreq + 10)
    # Show plot
    plt.title(title)
    plt.show()
    return

### Execute the circuit

In [ ]:
#
# Grover's algorithm
#


q = QuantumRegister(3 , 'x')
y = QuantumRegister(1 , 'y')
anc = QuantumRegister(1 , 'a')
c = ClassicalRegister(3 , 'c')
qc = QuantumCircuit(q, y, anc, c)

qc.x(y[0])
qc.h(q[0])
qc.h(q[1])
qc.h(q[2])
qc.h(y[0])

grover_oracle (qc, q[0], q[1], q[2], anc[0], y[0])
grover_diffusion_operator (qc, q[0], q[1], q[2], y[0])

grover_oracle (qc, q[0], q[1], q[2], anc[0], y[0])
grover_diffusion_operator (qc, q[0], q[1], q[2], y[0])

grover_oracle (qc, q[0], q[1], q[2], anc[0], y[0])
grover_diffusion_operator (qc, q[0], q[1], q[2], y[0])

qc.measure(q[0],c[0])
qc.measure(q[1],c[1])
qc.measure(q[2],c[2])

job = backend.run(qc, shots=shots)
job_monitor(job)
result = job.result()
counts = result.get_counts()
print (counts)

del q, y, anc
del c
del qc
del result
del job

plot_histogram(counts, "")